# 11. 6차 사전 등록 (김유환): 우세 모달리티 억제 + 반사실 이미지 활용도 (s2)

`configs/prereg_yh_balance.yaml` 대로 실행한다. 인코더 4종 × {기준, 이미지 드롭아웃, 텍스트 드롭아웃, 이미지만} 48회(약 20~25분, 평균 대체 추천 포함) → 분석(약 10분). 텍스트만(C0)은 5차(10 노트북) 결과를 재사용한다.

In [ ]:
# @title 0. 공통 준비 (모든 노트북 동일)
import sys
if any(m.startswith("msrs_paper") for m in sys.modules):
    raise RuntimeError("이전에 불러온 코드가 메모리에 남아 있습니다. 메뉴 '런타임 → 세션 다시 시작' 후 이 셀부터 다시 실행하세요.")
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

REPO_URL = "https://github.com/yuhwani/multimodal-recsys.git"  # upstream에 머지되면 3sunghynix 주소로 변경
BRANCH = "main"
!rm -rf /content/repo && git clone -q -b {BRANCH} {REPO_URL} /content/repo
!pip -q install -r /content/repo/paper/requirements.txt
!cd /content/repo && git log -1 --format="코드 커밋: %h %s (%cd)"

import sys
sys.path.insert(0, "/content/repo/paper")
from msrs_paper.config import load_all, Paths
cfg, exps, plan = load_all()
DRIVE_ROOT = cfg["drive_root"]  # 공유 폴더 바로가기 경로가 다르면 이 줄만 바꾼다
paths = Paths(DRIVE_ROOT).make_dirs()
print("드라이브 폴더:", paths.root)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# @title 1. 준비물 · 사전 등록 확인 · 진행 현황
from msrs_paper.utils import verify_manifest
from msrs_paper.config import load_yaml
m = verify_manifest(paths)
prereg = load_yaml("prereg_yh_balance")
assert prereg["status"] == "final" and (paths.split_dir("s2") / "stats.json").exists()
print(f"준비물 {len(m['files'])}개 일치 · 사전 등록 {prereg['registered']}")
def status():
    b = exps["groups"]["YH4"][0]
    runs = [(c, s) for s in b["seeds"] for c in b["conds"]]
    done = [r for r in runs if (paths.run_dir("YH4", "s2", f"{r[0]}_seed{r[1]}") / "run_info.json").exists()]
    c0 = [paths.run_dir("YH3_ENC", "s2", f"C0_seed{s}") / "run_info.json" for s in (42, 43, 44)]
    print(f"묶음 YH4: 전체 {len(runs)}회, 완료 {len(done)}회 · 재사용 텍스트만 C0 (YH3_ENC): {sum(f.exists() for f in c0)} / 3")
status()

In [ ]:
# @title 2. 학습 48회 (평균 대체 추천 포함, 약 20~25분)
import time
from msrs_paper import two_tower
from msrs_paper.utils import now
log_file = paths.logs / "yh_balance.log"
def log(msg):
    print(msg)
    with open(log_file, "a", encoding="utf-8") as f:
        f.write(f"{now()} {msg}\n")
t0 = time.time()
two_tower.run_group(cfg, exps, paths, "YH4", log=log)
log(f"6차 학습 완료 ({(time.time() - t0) / 60:.1f}분)")
status()

In [ ]:
# @title 3. 분석: 확증 H10 · H11 + 인코더 × 처방 효과 (약 10분)
from msrs_paper import analysis_yh4
confirmatory, effects, dominance, util = analysis_yh4.run(cfg, exps, paths, log=log)
effects[effects["segment"] == "all"][["encoder", "treatment", "R_base", "R_treat", "delta", "ci_low", "ci_high"]]

In [ ]:
# @title 4. 우세 모달리티 · 반사실 활용도 · 구간별 효과 (탐색적)
display(dominance)
display(util.pivot_table(index=["encoder", "model"], columns="measure", values="utilization").round(3))
display(effects[effects["segment"].isin(["established", "new_item"])][["encoder", "treatment", "segment", "delta", "ci_low", "ci_high"]])